# Building a GPT-Style Language Model from ScratchThis notebook provides a complete, step-by-step implementation of a GPT-style Large Language Model (LLM) using PyTorch. We'll build everything from the ground up, starting with basic components and progressing to a full training pipeline.## What You'll Learn1. **Tokenization**: Converting text into numbers that models can process2. **Embeddings**: Representing tokens as dense vectors3. **Positional Encoding**: Adding position information to embeddings4. **Self-Attention**: The core mechanism that lets models understand context5. **Multi-Head Attention**: Multiple attention mechanisms working in parallel6. **Transformer Blocks**: Complete decoder blocks with residual connections7. **GPT Architecture**: Putting it all together8. **Training**: Pretraining and fine-tuning the model## PrerequisitesBasic understanding of:- Python programming- Neural networks (what they are, how they learn)- PyTorch basics (tensors, modules)Let's begin!

## 1. Setup and ImportsFirst, we'll import all necessary libraries. Each import serves a specific purpose:- **torch**: The main PyTorch library for building neural networks- **torch.nn**: Contains neural network layers and functions- **transformers**: Hugging Face library for tokenizers- **datasets**: For loading training data- **math**: For mathematical operations (cosine scheduling)- **re**: Regular expressions for text cleaning- **os**: File system operations

In [ ]:
# Core PyTorch importsimport torchfrom torch import nnfrom torch.utils.data import Dataset, DataLoader# Hugging Face libraries for tokenization and datasetsfrom transformers import GPT2Tokenizerfrom datasets import load_dataset# Standard library importsimport mathimport reimport os# Set random seeds for reproducibility# This ensures that our results are consistent across runstorch.manual_seed(123)if torch.cuda.is_available():    torch.cuda.manual_seed_all(123)# Check if GPU is available and set device accordingly# GPU training is much faster than CPU for deep learningdevice = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Using device: {device}")if torch.cuda.is_available():    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Hyperparameters ConfigurationHyperparameters control the model's architecture and training process. Let's define them all in one place:### Model Architecture Parameters:- **vocab_size**: Total number of unique tokens the model can understand (50,257 for GPT-2)- **embed_dim**: Size of token embeddings (not used in this simplified version)- **attention_dim**: Dimension of attention mechanism (256 = smaller model, faster training)- **num_heads**: Number of parallel attention mechanisms (8 heads look at different aspects)- **num_blocks**: Number of transformer decoder blocks stacked (8 layers deep)- **context_length**: Maximum sequence length the model can process (256 tokens)- **dropout_rate**: Probability of dropping neurons during training (0.1 = 10% dropout for regularization)

In [ ]:
# Model architecture hyperparametersvocab_size = 50257        # GPT-2 tokenizer vocabulary sizeembed_dim = 256           # Embedding dimension (not used in simplified version)attention_dim = 256       # Attention mechanism dimensionnum_heads = 8             # Number of attention headsnum_blocks = 8            # Number of transformer blockscontext_length = 256      # Maximum sequence lengthdropout_rate = 0.1        # Dropout probability for regularizationprint("Model Configuration:")print(f"  Vocabulary Size: {vocab_size:,}")print(f"  Attention Dimension: {attention_dim}")print(f"  Number of Heads: {num_heads}")print(f"  Number of Blocks: {num_blocks}")print(f"  Context Length: {context_length}")print(f"  Dropout Rate: {dropout_rate}")

## 3. Tokenization**What is Tokenization?**Neural networks can't directly process text - they need numbers. Tokenization converts text into numerical tokens:```"Hello world" → [15496, 995] → Model processes these numbers```We use GPT-2's tokenizer, which uses **Byte-Pair Encoding (BPE)**:- Common words become single tokens: "hello" → one token- Rare words split into subwords: "unhappiness" → ["un", "happiness"]- This balances vocabulary size with coverage### Helper Functions:- `text_to_token_ids`: Converts text string to list of token IDs- `token_ids_to_text`: Converts token IDs back to readable text

In [ ]:
# Initialize the GPT-2 tokenizer# GPT-2 tokenizer uses Byte-Pair Encoding (BPE) which efficiently handles# both common and rare words by breaking them into subword unitstokenizer = GPT2Tokenizer.from_pretrained("gpt2")# Set padding token (GPT-2 doesn't have one by default)# We use the end-of-sequence token for paddingtokenizer.pad_token = tokenizer.eos_tokendef text_to_token_ids(text, tokenizer, device):    """    Convert text string to tensor of token IDs.        Args:        text (str): Input text to tokenize        tokenizer: Tokenizer object (GPT2Tokenizer)        device: Device to place tensor on (CPU or GPU)        Returns:        torch.Tensor: Token IDs as a tensor of shape [1, sequence_length]    """    # Encode text to token IDs (list of integers)    encoded = tokenizer.encode(text, add_special_tokens=False)    # Convert to tensor and add batch dimension [1, seq_len]    encoded_tensor = torch.tensor(encoded).unsqueeze(0)    return encoded_tensor.to(device)def token_ids_to_text(token_ids, tokenizer):    """    Convert token IDs back to readable text.        Args:        token_ids (torch.Tensor): Tensor of token IDs        tokenizer: Tokenizer object (GPT2Tokenizer)        Returns:        str: Decoded text string    """    # Flatten tensor to 1D and convert to list    flat = token_ids.squeeze(0)    return tokenizer.decode(flat.tolist())# Example: Tokenize a sample sentencesample_text = "Hello, how are you today?"print(f"Original text: {sample_text}")# Convert to token IDstoken_ids = text_to_token_ids(sample_text, tokenizer, device)print(f"Token IDs shape: {token_ids.shape}")print(f"Token IDs: {token_ids}")# Convert back to textdecoded_text = token_ids_to_text(token_ids, tokenizer)print(f"Decoded text: {decoded_text}")# Show individual tokensprint("\nIndividual tokens:")for i, token_id in enumerate(token_ids[0].tolist()):    token_text = tokenizer.decode([token_id])    print(f"  {i}: {token_id:5d} → '{token_text}'")

## 4. Self-Attention Mechanism**What is Self-Attention?**Self-attention is the core innovation that makes transformers powerful. It allows each word to "look at" and gather information from all other words in the sequence.### The Intuition:Consider the sentence: "The cat sat on the mat because it was tired."When processing "it", the model needs to understand that "it" refers to "cat", not "mat". Self-attention computes how much each word should pay attention to every other word.### How It Works:```ASCII Diagram of Self-Attention:Input: [The, cat, sat]  (each word is an embedding vector)         ↓    ↓    ↓    [Query, Key, Value] projections (learned linear transformations)         ↓    ↓    ↓    Attention Scores = Query × Key^T / √d         ↓    Softmax (convert to probabilities)         ↓    Weighted Sum of Values         ↓    Output: Context-aware representations```### Key Components:1. **Query (Q)**: "What am I looking for?"2. **Key (K)**: "What do I contain?"3. **Value (V)**: "What information do I have?"### Causal Masking:For language modeling, we use **causal masking** - each position can only attend to previous positions, not future ones. This prevents the model from "cheating" by looking ahead.```Attention Matrix (before masking):     The  cat  satThe  [1.0  0.8  0.2]cat  [0.3  1.0  0.7]sat  [0.1  0.4  1.0]After Causal Masking:     The  cat  satThe  [1.0  -∞   -∞ ]  ← "The" can only see itselfcat  [0.3  1.0  -∞ ]  ← "cat" can see "The" and itselfsat  [0.1  0.4  1.0]  ← "sat" can see all previous words```

In [ ]:
class SelfAttention(torch.nn.Module):    """    Self-Attention mechanism with causal masking.        This is the core component that allows the model to weigh the importance    of different words when processing each word in a sequence.    """        def __init__(self, embed_dim, attention_dim, dropout=0.1):        """        Args:            embed_dim (int): Dimension of input embeddings            attention_dim (int): Dimension of attention space (Q, K, V)            dropout (float): Dropout probability for regularization        """        super().__init__()                # Linear projections for Query, Key, and Value        # These are learned transformations that project embeddings into attention space        self.W_query = torch.nn.Linear(embed_dim, attention_dim, bias=False)        self.W_key = torch.nn.Linear(embed_dim, attention_dim, bias=False)        self.W_value = torch.nn.Linear(embed_dim, attention_dim, bias=False)                # Dropout for regularization (randomly zeros out some attention weights)        self.dropout = torch.nn.Dropout(dropout)                # Store attention dimension for scaling        self.attention_dim = attention_dim        def forward(self, x):        """        Forward pass of self-attention.                Args:            x (torch.Tensor): Input tensor of shape [batch_size, seq_len, embed_dim]                Returns:            torch.Tensor: Output tensor of shape [batch_size, seq_len, attention_dim]        """        batch_size, seq_len, embed_dim = x.shape                # Step 1: Project input to Query, Key, Value spaces        # Each projection learns to extract different aspects of the input        queries = self.W_query(x)  # [B, T, A] - "What am I looking for?"        keys = self.W_key(x)       # [B, T, A] - "What do I contain?"        values = self.W_value(x)   # [B, T, A] - "What information do I have?"                # Step 2: Compute attention scores        # Multiply queries by keys to see how much each position should attend to others        # Transpose keys: [B, T, A] → [B, A, T]        attention_scores = queries @ keys.transpose(1, 2)  # [B, T, T]                # Step 3: Scale attention scores        # Divide by √d to prevent extremely large values that would make softmax saturate        # This keeps gradients stable during training        attention_scores = attention_scores / (self.attention_dim ** 0.5)                # Step 4: Apply causal mask        # Create a lower triangular matrix of ones (allows attending to past)        # Upper triangle is zero (prevents attending to future)        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))                # Replace zeros with -inf so softmax makes them zero probability        attention_scores = attention_scores.masked_fill(mask == 0, float('-inf'))                # Step 5: Convert scores to probabilities using softmax        # Each row sums to 1.0, representing attention distribution        attention_weights = torch.nn.functional.softmax(attention_scores, dim=-1)                # Apply dropout to attention weights (randomly zero out some connections)        attention_weights = self.dropout(attention_weights)                # Step 6: Compute weighted sum of values        # Multiply attention weights by values to get context-aware representations        output = attention_weights @ values  # [B, T, A]                return output# Example: Test self-attention with dummy dataprint("Testing Self-Attention:")print("=" * 50)# Create dummy input: batch_size=2, seq_len=4, embed_dim=256dummy_input = torch.randn(2, 4, 256).to(device)print(f"Input shape: {dummy_input.shape}")print(f"  Batch size: {dummy_input.shape[0]}")print(f"  Sequence length: {dummy_input.shape[1]}")print(f"  Embedding dimension: {dummy_input.shape[2]}")# Create self-attention moduleself_attn = SelfAttention(embed_dim=256, attention_dim=256, dropout=0.1).to(device)# Forward passoutput = self_attn(dummy_input)print(f"\nOutput shape: {output.shape}")print(f"  Same batch size: {output.shape[0]}")print(f"  Same sequence length: {output.shape[1]}")print(f"  Attention dimension: {output.shape[2]}")print("\n✓ Self-attention working correctly!")

## 5. Multi-Head Attention: The Group Chat in Your Model's Brain**Why Multiple Heads?**A single attention head might focus on one aspect (e.g., syntactic relationships). But language is complex! We need multiple perspectives:- **Head 1**: Might focus on subject-verb agreement- **Head 2**: Might track pronoun references- **Head 3**: Might capture semantic relationships- **Head 4**: Might learn positional patterns### The Architecture:```Input Embedding (256-dim)         |    ┌────┴────┬────────┬────────┐    ↓         ↓        ↓        ↓  Head 1   Head 2   Head 3   Head 4  ... (8 heads total)  (32-dim) (32-dim) (32-dim) (32-dim)    ↓         ↓        ↓        ↓    └────┬────┴────────┴────────┘         ↓   Concatenate (256-dim)```### Key Insight:The total attention dimension (256) is divided among heads (256 ÷ 8 = 32 per head). This allows:- **Parallel processing**: All heads compute simultaneously- **Diverse representations**: Each head learns different patterns- **Efficient computation**: Same total parameters as single large head

In [ ]:
class MultiHeadAttention(torch.nn.Module):    """    Multi-Head Attention: Multiple attention mechanisms working in parallel.        This allows the model to jointly attend to information from different    representation subspaces at different positions.    """        def __init__(self, num_heads, embed_dim, attention_dim, dropout=0.1):        """        Args:            num_heads (int): Number of parallel attention heads            embed_dim (int): Dimension of input embeddings            attention_dim (int): Total attention dimension (split among heads)            dropout (float): Dropout probability        """        super().__init__()                # Calculate dimension per head        # If attention_dim=256 and num_heads=8, each head gets 32 dimensions        self.head_size = attention_dim // num_heads                # Create a list of attention heads        # ModuleList ensures PyTorch tracks all heads for training        self.heads = torch.nn.ModuleList()        for i in range(num_heads):            # Each head is a separate SelfAttention module            self.heads.append(                SelfAttention(                    embed_dim=embed_dim,                    attention_dim=self.head_size,  # Each head gets a slice                    dropout=dropout                )            )        def forward(self, x):        """        Forward pass through all attention heads.                Args:            x (torch.Tensor): Input of shape [batch_size, seq_len, embed_dim]                Returns:            torch.Tensor: Output of shape [batch_size, seq_len, attention_dim]        """        # Collect outputs from all heads        head_outputs = []                # Process input through each head independently        for head in self.heads:            # Each head produces [B, T, head_size] output            head_outputs.append(head(x))                # Concatenate all head outputs along the feature dimension        # If we have 8 heads with 32 dims each: [B, T, 32] × 8 → [B, T, 256]        concatenated = torch.cat(head_outputs, dim=2)                return concatenated# Example: Test multi-head attentionprint("Testing Multi-Head Attention:")print("=" * 50)# Create dummy inputdummy_input = torch.randn(2, 4, 256).to(device)print(f"Input shape: {dummy_input.shape}")# Create multi-head attention with 8 headsmulti_head_attn = MultiHeadAttention(    num_heads=8,    embed_dim=256,    attention_dim=256,    dropout=0.1).to(device)print(f"\nNumber of heads: 8")print(f"Dimension per head: {256 // 8} (total 256 ÷ 8 heads)")# Forward passoutput = multi_head_attn(dummy_input)print(f"\nOutput shape: {output.shape}")print(f"  Each head contributes {256 // 8} dimensions")print(f"  Total: 8 heads × {256 // 8} = {output.shape[2]} dimensions")print("\n✓ Multi-head attention working correctly!")

## 6. Feed-Forward Networks**What Happens After Attention?**After attention gathers context from other positions, we need to process this information. That's where the feed-forward network comes in.### Architecture:```Input (256-dim)      ↓  Linear Layer (up-projection)  256 → 1024 dimensions      ↓  GELU Activation  (smooth, non-linear transformation)      ↓  Linear Layer (down-projection)  1024 → 256 dimensions      ↓Output (256-dim)```### Why This Design?1. **Up-projection (4x expansion)**: Creates a high-dimensional space where the model can learn complex transformations2. **GELU activation**: Introduces non-linearity (without it, stacking layers would be pointless)3. **Down-projection**: Compresses back to original dimension### GELU vs ReLU:- **ReLU**: Hard cutoff at zero (x if x > 0, else 0)- **GELU**: Smooth, probabilistic activation that works better for transformersThink of it as: "Expand → Transform → Compress"

In [ ]:
class FeedForward(torch.nn.Module):    """    Position-wise Feed-Forward Network.        Applied to each position independently and identically.    Consists of two linear transformations with GELU activation.    """        def __init__(self, attention_dim):        """        Args:            attention_dim (int): Dimension of input/output        """        super().__init__()                # Up-projection: Expand to 4x the dimension        # This creates a high-dimensional space for complex transformations        # Example: 256 → 1024 dimensions        self.up = torch.nn.Linear(attention_dim, attention_dim * 4)                # GELU activation: Gaussian Error Linear Unit        # Smoother than ReLU, works better for transformers        # Formula: x * Φ(x) where Φ is cumulative distribution function        self.gelu = torch.nn.GELU()                # Down-projection: Compress back to original dimension        # Example: 1024 → 256 dimensions        self.down = torch.nn.Linear(attention_dim * 4, attention_dim)        def forward(self, x):        """        Forward pass through feed-forward network.                Args:            x (torch.Tensor): Input of shape [batch_size, seq_len, attention_dim]                Returns:            torch.Tensor: Output of same shape as input        """        # Step 1: Expand to higher dimension        x = self.up(x)  # [B, T, attention_dim] → [B, T, attention_dim*4]                # Step 2: Apply non-linear activation        x = self.gelu(x)  # Smooth non-linearity                # Step 3: Project back to original dimension        x = self.down(x)  # [B, T, attention_dim*4] → [B, T, attention_dim]                return x# Example: Test feed-forward networkprint("Testing Feed-Forward Network:")print("=" * 50)# Create dummy inputdummy_input = torch.randn(2, 4, 256).to(device)print(f"Input shape: {dummy_input.shape}")print(f"  Dimension: 256")# Create feed-forward networkffn = FeedForward(attention_dim=256).to(device)# Forward passoutput = ffn(dummy_input)print(f"\nOutput shape: {output.shape}")print(f"  Dimension: {output.shape[2]} (same as input)")# Show intermediate dimensionsprint("\nInternal transformations:")print(f"  1. Input:  256 dimensions")print(f"  2. Up:     256 → {256 * 4} dimensions (4x expansion)")print(f"  3. GELU:   Apply smooth activation")print(f"  4. Down:   {256 * 4} → 256 dimensions (back to original)")print("\n✓ Feed-forward network working correctly!")

## 7. The Decoder Block with Residual Connections**Putting It Together: The Transformer Block**Now we combine multi-head attention and feed-forward networks into a complete decoder block. But there are two crucial additions:### 1. Residual Connections (Skip Connections)```Input (x)  |  |----→ (skip connection)  ↓                      |Layer                    |  ↓                      |Output ←------ ADD ------┘```**Why?** Deep networks suffer from vanishing gradients. Residual connections create "highways" for gradients to flow backward during training.### 2. Layer Normalization**What it does:** Normalizes activations to have mean=0 and variance=1**Why?** Keeps activations stable as we stack many layers. Unlike BatchNorm (normalizes across batch), LayerNorm normalizes across features of a single example.### Complete Decoder Block Architecture:```Input  ↓LayerNorm  ↓Multi-Head Attention  ↓Add (residual) ←---┐  ↓                |LayerNorm          |  ↓                |Feed-Forward       |  ↓                |Add (residual) ←---┘  ↓Output```This is called **Pre-LN** (Pre-Layer Normalization) architecture, which is more stable than the original Post-LN design.

In [ ]:
class Decoder(torch.nn.Module):    """    Transformer Decoder Block.        Combines multi-head attention and feed-forward network with    residual connections and layer normalization.    """        def __init__(self, num_heads, embed_dim, attention_dim, dropout=0.1):        """        Args:            num_heads (int): Number of attention heads            embed_dim (int): Embedding dimension            attention_dim (int): Attention dimension            dropout (float): Dropout probability        """        super().__init__()                # Multi-head attention layer        self.masked_multihead = MultiHeadAttention(            num_heads, embed_dim, attention_dim, dropout        )                # Feed-forward network        self.feed_forward = FeedForward(attention_dim)                # Layer normalization layers        # n1: Normalizes before attention        # n2: Normalizes before feed-forward        self.n1 = torch.nn.LayerNorm(attention_dim)        self.n2 = torch.nn.LayerNorm(attention_dim)        def forward(self, x):        """        Forward pass through decoder block.                Args:            x (torch.Tensor): Input of shape [batch_size, seq_len, attention_dim]                Returns:            torch.Tensor: Output of same shape as input        """        # First sub-layer: Multi-head attention with residual connection        # Step 1: Normalize input        normalized = self.n1(x)                # Step 2: Apply multi-head attention        attention_output = self.masked_multihead(normalized)                # Step 3: Add residual connection (skip connection)        # This allows gradients to flow directly backward        x = attention_output + x                # Second sub-layer: Feed-forward with residual connection        # Step 4: Normalize        normalized = self.n2(x)                # Step 5: Apply feed-forward network        ff_output = self.feed_forward(normalized)                # Step 6: Add residual connection        x = ff_output + x                return x# Example: Test decoder blockprint("Testing Decoder Block:")print("=" * 50)# Create dummy inputdummy_input = torch.randn(2, 4, 256).to(device)print(f"Input shape: {dummy_input.shape}")# Create decoder blockdecoder = Decoder(    num_heads=8,    embed_dim=256,    attention_dim=256,    dropout=0.1).to(device)# Forward passoutput = decoder(dummy_input)print(f"Output shape: {output.shape}")print(f"  Shape preserved: {output.shape == dummy_input.shape}")# Show the flowprint("\nData flow through decoder block:")print("  1. Input → LayerNorm → Multi-Head Attention")print("  2. Add residual connection (skip)")print("  3. → LayerNorm → Feed-Forward Network")print("  4. Add residual connection (skip)")print("  5. → Output")print("\n✓ Decoder block working correctly!")

## 8. Complete GPT Model Architecture**Putting It All Together: The Full GPT Model**Now we assemble all components into a complete GPT-style language model.### Architecture Overview:```Input Token IDs [101, 102, 103, ...]         ↓    Token Embedding (lookup table)    [101] → [0.2, -0.5, 0.8, ...] (256-dim vector)         ↓    Positional Embedding (position info)    Position 0 → [0.1, 0.3, -0.2, ...]         ↓    Add: Token + Position Embeddings         ↓    ┌─────────────────┐    │ Decoder Block 1 │    └─────────────────┘         ↓    ┌─────────────────┐    │ Decoder Block 2 │    └─────────────────┘         ↓        ...         ↓    ┌─────────────────┐    │ Decoder Block 8 │    └─────────────────┘         ↓    Layer Normalization         ↓    Linear (LM Head)    256-dim → 50,257-dim (vocab size)         ↓    Logits [0.2, -1.5, 3.2, ...] (one per vocab token)         ↓    Softmax → Probabilities```### Key Components:1. **Token Embedding**: Converts token IDs to dense vectors2. **Positional Embedding**: Adds position information ("where am I in the sequence?")3. **Decoder Blocks**: Stack of transformer blocks (8 in our case)4. **LM Head**: Projects to vocabulary size for next-token prediction### Why Positional Embeddings?Attention has no inherent notion of position. Without positional embeddings:- "The cat sat on the mat" = "mat the on sat cat The" (same attention!)Positional embeddings are learned vectors that encode position information.

In [ ]:
class GPT(nn.Module):    """    GPT-style Language Model.        A decoder-only transformer that predicts the next token given previous tokens.    """        def __init__(self, num_heads, vocab_size, embed_dim, attention_dim,                 num_blocks, context_length, dropout_rate):        """        Args:            num_heads (int): Number of attention heads per block            vocab_size (int): Size of vocabulary (50,257 for GPT-2)            embed_dim (int): Embedding dimension (not used in simplified version)            attention_dim (int): Attention dimension            num_blocks (int): Number of decoder blocks to stack            context_length (int): Maximum sequence length            dropout_rate (float): Dropout probability        """        super().__init__()                # Token embedding layer        # Converts token IDs to dense vectors        # Example: token 101 → [0.2, -0.5, 0.8, ..., 0.1] (256-dim vector)        self.embedding = nn.Embedding(vocab_size, attention_dim)                # Positional embedding layer        # Learns position-specific patterns        # Position 0 → [0.1, 0.3, -0.2, ...]        # Position 1 → [0.5, -0.1, 0.4, ...]        self.positional_embedding = nn.Embedding(context_length, attention_dim)                # Stack of decoder blocks        # Each block contains multi-head attention + feed-forward        self.decoders = nn.ModuleList([            Decoder(num_heads, attention_dim, attention_dim, dropout_rate)            for _ in range(num_blocks)        ])                # Final layer normalization        # Stabilizes the output before projection        self.exit_norm = nn.LayerNorm(attention_dim)                # Language modeling head        # Projects from attention_dim to vocab_size        # Output: logits for each token in vocabulary        self.linear = nn.Linear(attention_dim, vocab_size)        def forward(self, context):        """        Forward pass through the model.                Args:            context (torch.Tensor): Input token IDs of shape [batch_size, seq_len]                Returns:            torch.Tensor: Logits of shape [batch_size, seq_len, vocab_size]        """        # Step 1: Get token embeddings        # [B, T] → [B, T, attention_dim]        embeddings = self.embedding(context)                # Step 2: Create position indices        # [0, 1, 2, ..., seq_len-1]        context_len = context.shape[1]        position = torch.arange(context_len, device=context.device).unsqueeze(0)                # Step 3: Get positional embeddings        # [1, T] → [1, T, attention_dim]        position_embeddings = self.positional_embedding(position)                # Step 4: Add token and positional embeddings        # This gives each token both content and position information        e = embeddings + position_embeddings                # Step 5: Pass through all decoder blocks        # Each block refines the representations        for decoder in self.decoders:            e = decoder(e)                # Step 6: Final layer normalization        e = self.exit_norm(e)                # Step 7: Project to vocabulary size        # [B, T, attention_dim] → [B, T, vocab_size]        logits = self.linear(e)                return logits# Example: Create and test the modelprint("Creating GPT Model:")print("=" * 50)model = GPT(    num_heads=num_heads,    vocab_size=vocab_size,    embed_dim=embed_dim,    attention_dim=attention_dim,    num_blocks=num_blocks,    context_length=context_length,    dropout_rate=dropout_rate).to(device)# Count parameterstotal_params = sum(p.numel() for p in model.parameters())trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print(f"Model created successfully!")print(f"  Total parameters: {total_params:,}")print(f"  Trainable parameters: {trainable_params:,}")print(f"  Model size: ~{total_params * 4 / 1024 / 1024:.2f} MB (float32)")# Test forward passdummy_input = torch.randint(0, vocab_size, (2, 10)).to(device)print(f"\nTest input shape: {dummy_input.shape}")print(f"  Batch size: {dummy_input.shape[0]}")print(f"  Sequence length: {dummy_input.shape[1]}")output = model(dummy_input)print(f"\nOutput shape: {output.shape}")print(f"  Batch size: {output.shape[0]}")print(f"  Sequence length: {output.shape[1]}")print(f"  Vocabulary size: {output.shape[2]}")print("\n✓ GPT model working correctly!")

## 9. Text Generation Functions**How Does the Model Generate Text?**The model outputs logits (raw scores) for each token in the vocabulary. To generate text, we:1. **Convert logits to probabilities** using softmax2. **Sample the next token** from the probability distribution3. **Append it to the context** and repeat### Sampling Strategies:- **Greedy**: Always pick the highest probability token (deterministic, boring)- **Temperature**: Scale logits to control randomness (higher = more random)- **Top-k**: Only sample from the k most likely tokens (prevents nonsense)### Temperature Effect:```Original logits: [2.0, 1.0, 0.5]Temperature = 0.5 (more confident):  Scaled: [4.0, 2.0, 1.0]  Probabilities: [0.84, 0.12, 0.04]  ← Peaked distributionTemperature = 2.0 (more random):  Scaled: [1.0, 0.5, 0.25]  Probabilities: [0.46, 0.28, 0.26]  ← Flatter distribution```

In [ ]:
def top_k_logits(logits, k):    """    Filter logits to only keep top k values.        Args:        logits (torch.Tensor): Logits of shape [batch_size, vocab_size]        k (int): Number of top values to keep        Returns:        torch.Tensor: Filtered logits with same shape    """    # Get top k values and their indices    v, ix = torch.topk(logits, k)        # Clone logits and set all values below top-k to -inf    out = logits.clone()    out[out < v[:, [-1]]] = float('-inf')        return outdef generate(model, max_new_tokens, context, context_length, temperature=1.0, top_k=None):    """    Generate text using the model.        Args:        model: The GPT model        max_new_tokens (int): Number of tokens to generate        context (torch.Tensor): Starting context [1, seq_len]        context_length (int): Maximum context length        temperature (float): Sampling temperature (higher = more random)        top_k (int): If set, only sample from top k tokens        Returns:        torch.Tensor: Generated sequence including original context    """    model.eval()  # Set to evaluation mode        with torch.no_grad():  # Don't compute gradients        for _ in range(max_new_tokens):            # Truncate context if it exceeds maximum length            if context.shape[1] > context_length:                context = context[:, -context_length:]                        # Get model predictions            logits = model(context)  # [B, T, V]                        # Focus on the last position (next token prediction)            logits = logits[:, -1, :]  # [B, V]                        # Apply temperature scaling            # Higher temperature = more random, lower = more deterministic            logits = logits / max(temperature, 1e-3)                        # Apply top-k filtering if specified            if top_k is not None:                logits = top_k_logits(logits, top_k)                        # Check for invalid values            if torch.isnan(logits).any() or torch.isinf(logits).any():                raise ValueError("Logits contain NaN or Inf")                        # Convert logits to probabilities            probabilities = nn.functional.softmax(logits, dim=-1)                        # Clamp probabilities to avoid numerical issues            probabilities = torch.clamp(probabilities, min=1e-9, max=1.0)                        # Sample next token from probability distribution            next_token = torch.multinomial(probabilities, 1)  # [B, 1]                        # Append to context            context = torch.cat((context, next_token), dim=1)        return context# Example: Generate text (will be random before training)print("Testing Text Generation:")print("=" * 50)start_context = "I want something"print(f"Starting context: '{start_context}'")# Convert text to token IDscontext_ids = text_to_token_ids(start_context, tokenizer, device)print(f"Context token IDs: {context_ids}")# Generate tokensgenerated_ids = generate(    model=model,    context=context_ids,    max_new_tokens=10,    context_length=context_length,    temperature=1.0,    top_k=50)# Convert back to textgenerated_text = token_ids_to_text(generated_ids, tokenizer)print(f"\nGenerated text (untrained model - will be random):")print(f"'{generated_text}'")print("\n✓ Text generation working correctly!")print("Note: Output is random because model is not trained yet.")

## 10. Weight Initialization**Why Initialize Weights?**Neural networks need good starting points. Random initialization can lead to:- Vanishing gradients (weights too small)- Exploding gradients (weights too large)- Slow convergence### GPT-2 Style Initialization:- **Linear layers**: Normal distribution with mean=0, std=0.02- **Embeddings**: Normal distribution with mean=0, std=0.02- **Layer Norm**: Weights=1, Biases=0This ensures the model starts from a reasonable state.

In [ ]:
def initialize_weights(module):    """    Initialize model weights using GPT-2 style initialization.        Args:        module: PyTorch module to initialize    """    if isinstance(module, nn.Linear):        # Initialize linear layer weights from normal distribution        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)        # Initialize biases to zero if they exist        if module.bias is not None:            torch.nn.init.zeros_(module.bias)        elif isinstance(module, nn.Embedding):        # Initialize embedding weights from normal distribution        torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)        elif isinstance(module, nn.LayerNorm):        # Initialize layer norm weights to 1 and biases to 0        torch.nn.init.ones_(module.weight)        torch.nn.init.zeros_(module.bias)# Apply initialization to the modelprint("Initializing model weights...")model.apply(initialize_weights)print("✓ Weights initialized successfully!")# Show some weight statisticsprint("\nWeight statistics after initialization:")for name, param in model.named_parameters():    if 'weight' in name:        print(f"  {name:50s} mean={param.mean().item():7.4f}, std={param.std().item():7.4f}")        break  # Just show first few

## 11. Data Preparation for Training**What Data Do We Need?**For pretraining, we need a large corpus of text. We'll use the IMDb dataset as an example.### Data Format:The model learns by predicting the next token:```Input:  [The, cat, sat, on, the]Target: [cat, sat, on, the, mat]```Each input is shifted by one position to create the target.### Sliding Window:We use a sliding window to create overlapping sequences:```Text: "The cat sat on the mat and slept"Window 1: [The, cat, sat, on]Window 2: [cat, sat, on, the]Window 3: [sat, on, the, mat]...```This maximizes the training data from our corpus.

In [ ]:
# Load and prepare datasetprint("Loading IMDb dataset...")print("=" * 50)# Load dataset from Hugging Faceds = load_dataset("stanfordnlp/imdb")print(f"Dataset loaded:")print(f"  Train samples: {len(ds['train']):,}")print(f"  Test samples: {len(ds['test']):,}")def keep_english_only(text):    """Remove non-ASCII characters from text."""    return re.sub(r"[^\x00-\x7F]+", "", text)def combine_and_clean(text_list):    """    Clean and combine a list of texts into one string.        Args:        text_list (list): List of text strings        Returns:        str: Combined and cleaned text    """    # Keep only English (ASCII) characters    cleaned_list = [keep_english_only(t) for t in text_list]        # Combine into one string    combined = " ".join(cleaned_list)        # Remove extra spaces/newlines    combined = re.sub(r'\s+', ' ', combined).strip()        return combined# Create combined text dataprint("\nCleaning and combining text...")train_text_data = combine_and_clean(ds['train']['text'][:1000])  # Use subset for demotest_text_data = combine_and_clean(ds['test']['text'][:200])# Show statisticstotal_characters = len(train_text_data)total_tokens = len(tokenizer.encode(train_text_data))print(f"\nTraining data statistics:")print(f"  Characters: {total_characters:,}")print(f"  Tokens: {total_tokens:,}")print(f"  Compression ratio: {total_characters / total_tokens:.2f} chars/token")# Show sampleprint(f"\nSample text (first 200 chars):")print(f"'{train_text_data[:200]}...'")print("\n✓ Data prepared successfully!")

## 12. Custom Dataset and DataLoader**Creating Training Batches**We need to convert our text into batches of input-target pairs.### CustomDataset Class:- Tokenizes the entire text- Creates overlapping windows using a sliding window approach- Each window becomes one training example- Stride controls overlap (stride=max_length means no overlap)### DataLoader:- Batches multiple examples together- Shuffles data for better training- Handles parallel data loading

In [ ]:
class CustomDataset(Dataset):    """    Custom dataset for language modeling.        Creates input-target pairs using a sliding window over tokenized text.    """        def __init__(self, txt, tokenizer, max_length, stride):        """        Args:            txt (str): Input text            tokenizer: Tokenizer object            max_length (int): Length of each sequence            stride (int): Step size for sliding window        """        self.input_ids = []        self.target_ids = []                # Tokenize the entire text        token_ids = tokenizer.encode(txt, add_special_tokens=False)                # Use a sliding window to chunk the data into overlapping sequences        for i in range(0, len(token_ids) - max_length, stride):            # Input: tokens at positions [i, i+1, ..., i+max_length-1]            input_chunk = token_ids[i:i + max_length]                        # Target: tokens at positions [i+1, i+2, ..., i+max_length]            # (shifted by one position)            target_chunk = token_ids[i + 1: i + max_length + 1]                        self.input_ids.append(torch.tensor(input_chunk))            self.target_ids.append(torch.tensor(target_chunk))        def __len__(self):        """Return the number of samples in the dataset."""        return len(self.input_ids)        def __getitem__(self, idx):        """Get a single sample (input, target) pair."""        return self.input_ids[idx], self.target_ids[idx]def create_encoded_dataloader(txt, tokenizer, batch_size=4, max_length=128,                               stride=128, shuffle=True, drop_last=True, num_workers=0):    """    Create a DataLoader for the dataset.        Args:        txt (str): Input text        tokenizer: Tokenizer object        batch_size (int): Number of samples per batch        max_length (int): Length of each sequence        stride (int): Step size for sliding window        shuffle (bool): Whether to shuffle data        drop_last (bool): Whether to drop incomplete batches        num_workers (int): Number of parallel workers        Returns:        DataLoader: PyTorch DataLoader object    """    # Create dataset    dataset = CustomDataset(txt, tokenizer, max_length, stride)        # Create dataloader    dataloader = DataLoader(        dataset,        batch_size=batch_size,        shuffle=shuffle,        drop_last=drop_last,        num_workers=num_workers,        pin_memory=True    )        return dataloader# Create dataloadersprint("Creating dataloaders...")print("=" * 50)train_dataloader = create_encoded_dataloader(    train_text_data,    tokenizer=tokenizer,    batch_size=2,    max_length=context_length,    stride=context_length,    shuffle=True,    drop_last=True)test_dataloader = create_encoded_dataloader(    test_text_data,    tokenizer=tokenizer,    batch_size=2,    max_length=context_length,    stride=context_length,    shuffle=False,    drop_last=True)print(f"Train dataloader: {len(train_dataloader)} batches")print(f"Test dataloader: {len(test_dataloader)} batches")# Show a sample batchprint("\nSample batch:")for inp, tgt in train_dataloader:    print(f"  Input shape: {inp.shape}")    print(f"  Target shape: {tgt.shape}")    print(f"\n  First input sequence (first 10 tokens):")    print(f"    Token IDs: {inp[0, :10].tolist()}")    print(f"    Text: '{tokenizer.decode(inp[0, :10].tolist())}'")    print(f"\n  Corresponding target (first 10 tokens):")    print(f"    Token IDs: {tgt[0, :10].tolist()}")    print(f"    Text: '{tokenizer.decode(tgt[0, :10].tolist())}'")    breakprint("\n✓ Dataloaders created successfully!")

## 13. Loss Calculation and Evaluation**Cross-Entropy Loss for Language Modeling**LLMs perform multi-class classification at each position - predicting which token comes next from the entire vocabulary.### How Cross-Entropy Works:```Example:  Input tokens:  ["The", "cat", "sat", "on", "the"]  Target tokens: ["cat", "sat", "on", "the", "mat"]For position 0:  Model predicts: P("cat") = 0.90, P("dog") = 0.05, ...  Target: "cat"  Loss = -log(0.90) ≈ 0.105  (low loss, good prediction)For position 1:  Model predicts: P("sat") = 0.10, P("ran") = 0.30, ...  Target: "sat"  Loss = -log(0.10) = 2.302  (high loss, bad prediction)Average loss across all positions = overall model performance```### Why Cross-Entropy?- Rewards high probability for correct tokens- Heavily penalizes confident wrong predictions- Differentiable (can compute gradients for training)

In [ ]:
# Define loss functioncriterion = nn.CrossEntropyLoss()def calc_loss_batch(input_batch, target_batch, model, device):    """    Calculate loss for a single batch.        Args:        input_batch (torch.Tensor): Input token IDs [B, T]        target_batch (torch.Tensor): Target token IDs [B, T]        model: The GPT model        device: Device to run on        Returns:        torch.Tensor: Scalar loss value    """    # Move data to device    input_batch = input_batch.to(device, non_blocking=True)    target_batch = target_batch.to(device, non_blocking=True)        # Forward pass    logits = model(input_batch)  # [B, T, V]        # Get dimensions    B, T, V = logits.shape        # Reshape for cross-entropy loss    # CrossEntropyLoss expects: (N, C) for inputs and (N,) for targets    # where N = batch_size * seq_len, C = vocab_size    loss = criterion(        logits.view(B * T, V),      # [B*T, V]        target_batch.view(B * T)     # [B*T]    )        return loss@torch.no_grad()def calc_loss_loader(data_loader, model, device, num_batches=None):    """    Calculate average loss over a dataloader.        Args:        data_loader: DataLoader object        model: The GPT model        device: Device to run on        num_batches (int): Number of batches to evaluate (None = all)        Returns:        float: Average loss    """    if len(data_loader) == 0:        return float("nan")        model.eval()  # Set to evaluation mode    total_loss = 0.0        # Determine number of batches to process    num_batches = len(data_loader) if num_batches is None else min(num_batches, len(data_loader))        # Calculate loss for each batch    for i, (inp, tgt) in enumerate(data_loader):        if i >= num_batches:            break        loss = calc_loss_batch(inp, tgt, model, device)        total_loss += loss.item()        model.train()  # Set back to training mode    return total_loss / num_batches@torch.no_grad()def evaluate_model(model, train_loader, val_loader, device, eval_iter=1):    """    Evaluate model on both train and validation sets.        Args:        model: The GPT model        train_loader: Training dataloader        val_loader: Validation dataloader        device: Device to run on        eval_iter (int): Number of batches to evaluate        Returns:        tuple: (train_loss, val_loss)    """    train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)    val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)    return train_loss, val_loss# Test loss calculationprint("Testing loss calculation...")print("=" * 50)# Get a sample batchfor inp, tgt in train_dataloader:    loss = calc_loss_batch(inp, tgt, model, device)    print(f"Sample batch loss: {loss.item():.4f}")    print(f"  (High loss is expected for untrained model)")    break# Evaluate on full datasettrain_loss, val_loss = evaluate_model(model, train_dataloader, test_dataloader, device, eval_iter=5)print(f"\nInitial losses (untrained model):")print(f"  Train loss: {train_loss:.4f}")print(f"  Val loss: {val_loss:.4f}")print("\n✓ Loss calculation working correctly!")

## 14. Learning Rate Scheduler**Why Adjust Learning Rate?**The learning rate controls how much we update weights during training:- **Too high**: Model diverges, loss explodes- **Too low**: Training is painfully slow- **Just right**: Fast convergence to good solution### Cosine Schedule with Warmup:```Learning Rate over Time:LR |     Warmup        Cosine Decay |      /\ |     /  \___ |    /       \___ |   /            \___ |  /                 \___ | /                      \___ |/                           \___ +---------------------------------> Steps 0    warmup              total_steps```### Two Phases:1. **Warmup (first 1500 steps)**:   - Linearly increase LR from 0 to max_lr   - Prevents early instability   - Helps model find good initial direction2. **Cosine Decay (remaining steps)**:   - Smoothly decrease LR following cosine curve   - Allows model to "settle" into local minimum   - Better final performance than constant LR

In [ ]:
class CosineWithWarmup(torch.optim.lr_scheduler._LRScheduler):    """    Learning rate scheduler with linear warmup and cosine decay.    """        def __init__(self, optimizer, warmup_steps, total_steps, base_lr, min_lr, last_epoch=-1):        """        Args:            optimizer: PyTorch optimizer            warmup_steps (int): Number of warmup steps            total_steps (int): Total number of training steps            base_lr (float): Maximum learning rate (after warmup)            min_lr (float): Minimum learning rate (end of decay)            last_epoch (int): Last epoch number        """        self.warmup_steps = max(1, warmup_steps)        self.total_steps = max(self.warmup_steps + 1, total_steps)        self.base_lr = base_lr        self.min_lr = min_lr        super().__init__(optimizer, last_epoch)        def get_lr(self):        """Calculate learning rate for current step."""        step = self.last_epoch + 1        lrs = []                for _ in self.base_lrs:            if step <= self.warmup_steps:                # Warmup phase: linear increase                lr = self.base_lr * step / self.warmup_steps            else:                # Cosine decay phase                progress = (step - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)                lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))                        lrs.append(lr)                return lrs# Visualize learning rate scheduleprint("Learning Rate Schedule:")print("=" * 50)# Create dummy optimizer and schedulerdummy_optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)scheduler = CosineWithWarmup(    dummy_optimizer,    warmup_steps=1500,    total_steps=10000,    base_lr=3e-4,    min_lr=3e-5)# Sample learning rates at different stepssample_steps = [0, 500, 1500, 3000, 5000, 7500, 10000]print("\nLearning rate at different steps:")for step in sample_steps:    # Simulate stepping to this point    for _ in range(step - scheduler.last_epoch - 1):        scheduler.step()    lr = scheduler.get_lr()[0]    print(f"  Step {step:5d}: LR = {lr:.6f}")print("\n✓ Scheduler working correctly!")

## 15. Training Loop**The Main Training Process**This is where the model actually learns! The training loop:1. **Forward pass**: Compute predictions2. **Calculate loss**: Compare predictions to targets3. **Backward pass**: Compute gradients4. **Update weights**: Adjust parameters using optimizer5. **Repeat**: Do this for many epochs### Key Training Techniques:**1. Gradient Clipping**```Without clipping:        With clipping:Gradient = 100.0    →    Gradient = 1.0 (clipped)Update = huge       →    Update = reasonableLoss = NaN          →    Loss = stable```**2. Early Stopping**- Monitor validation loss- Stop if no improvement for N epochs- Prevents overfitting and saves compute**3. AdamW Optimizer**- Adaptive learning rates per parameter- Momentum for faster convergence- Weight decay for regularization### Training Settings:- **Learning rate**: 3e-4 (standard for GPT)- **Batch size**: 32 (balance between speed and memory)- **Gradient clip**: 1.0 (prevent exploding gradients)- **Warmup steps**: 1500 (stabilize early training)- **Patience**: 50 (early stopping threshold)

In [ ]:
# Training settingssettings = {    "learning_rate": 3e-4,    "weight_decay": 0.1,        # Standard for GPT-style training    "num_epochs": 10,           # Reduced for demo (use 300+ for real training)    "batch_size": 32,    "warmup_steps": 1500,    "max_lr": 3e-4,    "min_lr": 3e-5,    "eval_freq": 200,           # Evaluate every N steps    "eval_iter": 20,            # Number of batches for evaluation    "gradient_clip": 1.0,    "patience": 50,    "min_improvement": 1e-4,    "print_interval": 1,    "generate_interval": 5}print("Training Settings:")print("=" * 50)for key, value in settings.items():    print(f"  {key:20s}: {value}")# Recreate dataloaders with correct batch sizetrain_dataloader = create_encoded_dataloader(    train_text_data,    tokenizer=tokenizer,    batch_size=settings["batch_size"],    max_length=context_length,    stride=context_length,    shuffle=True,    drop_last=True)test_dataloader = create_encoded_dataloader(    test_text_data,    tokenizer=tokenizer,    batch_size=settings["batch_size"],    max_length=context_length,    stride=context_length,    shuffle=False,    drop_last=True)print(f"\nDataloaders updated:")print(f"  Train batches: {len(train_dataloader)}")print(f"  Test batches: {len(test_dataloader)}")

In [ ]:
def train_model(    model,    train_loader,    val_loader,    device,    settings,    save_path="checkpoints/gpt_model.pt",):    """    Train the GPT model.        Args:        model: The GPT model        train_loader: Training dataloader        val_loader: Validation dataloader        device: Device to train on        settings (dict): Training settings        save_path (str): Path to save best model        Returns:        tuple: (train_losses, val_losses, tokens_seen_track)    """    # Set random seeds    torch.manual_seed(123)    if torch.cuda.is_available():        torch.cuda.manual_seed_all(123)        # Move model to device    model.to(device)        # Create optimizer    optimizer = torch.optim.AdamW(        model.parameters(),        lr=settings["learning_rate"],        weight_decay=settings["weight_decay"],        betas=(0.9, 0.95),  # GPT-style betas    )        # Create learning rate scheduler    total_steps = settings["num_epochs"] * len(train_loader)    scheduler = CosineWithWarmup(        optimizer,        warmup_steps=settings["warmup_steps"],        total_steps=total_steps,        base_lr=settings["max_lr"],        min_lr=settings["min_lr"],    )        # Tracking variables    train_losses, val_losses, tokens_seen_track = [], [], []    tokens_seen, global_step = 0, -1    best_val_loss, patience_counter = float("inf"), 0        print("\nStarting training...")    print("=" * 70)        # Training loop    for epoch in range(settings["num_epochs"]):        model.train()                for step, (inp, tgt) in enumerate(train_loader):            # Calculate loss            loss = calc_loss_batch(inp, tgt, model, device)                        # Backward pass            loss.backward()                        # Gradient clipping (prevent exploding gradients)            torch.nn.utils.clip_grad_norm_(model.parameters(), settings["gradient_clip"])                        # Update weights            optimizer.step()            optimizer.zero_grad(set_to_none=True)                        # Update learning rate            scheduler.step()                        # Update counters            global_step += 1            tokens_seen += inp.numel()                        # Evaluation            if global_step % settings["eval_freq"] == 0:                train_loss, val_loss = evaluate_model(                    model, train_loader, val_loader, device,                    eval_iter=settings["eval_iter"],                )                train_losses.append(train_loss)                val_losses.append(val_loss)                tokens_seen_track.append(tokens_seen)                                lr_now = optimizer.param_groups[0]["lr"]                print(f"Ep {epoch+1:2d} | step {global_step:06d} | lr {lr_now:.3e} "                      f"| train {train_loss:.3f} | val {val_loss:.3f}")                                # Early stopping check                if val_loss + settings["min_improvement"] < best_val_loss:                    best_val_loss = val_loss                    patience_counter = 0                                        # Save best model                    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)                    torch.save({                        "model_state": model.state_dict(),                        "optimizer_state": optimizer.state_dict(),                        "epoch": epoch,                        "global_step": global_step,                    }, save_path)                    print(f"  [Checkpoint saved: val_loss improved to {val_loss:.3f}]")                else:                    patience_counter += 1                    if patience_counter >= settings["patience"]:                        print(f"\nEarly stopping triggered after {patience_counter} evaluations without improvement.")                        return train_losses, val_losses, tokens_seen_track        print("\nTraining completed!")    return train_losses, val_losses, tokens_seen_track# Note: Actual training would take hours/days# This is a demonstration of the training loop structureprint("Training loop defined successfully!")print("\nNote: Full training would take significant time.")print("For demonstration purposes, you can run with reduced epochs.")

## 16. Fine-Tuning on Specific Data**From General to Specific**After pretraining on general text (IMDb reviews), we can fine-tune on specific data (e.g., Coldplay lyrics) to adapt the model's style.### Pretraining vs Fine-Tuning:**Pretraining:**- Large, diverse dataset (millions of tokens)- Learns general language patterns- High learning rate, many epochs- Goal: Understand grammar, vocabulary, world knowledge**Fine-Tuning:**- Small, specific dataset (thousands of tokens)- Adapts to particular style/domain- Low learning rate, few epochs- Goal: Match specific writing style or task### Fine-Tuning Settings:Key differences from pretraining:- **Lower learning rate** (1e-5 vs 3e-4): Preserve pretrained knowledge- **Fewer epochs** (5 vs 300): Prevent overfitting on small data- **Smaller batch size** (4 vs 32): Better for small datasets- **Reduced weight decay** (0.01 vs 0.1): Less regularization needed### Example: Coldplay LyricsAfter fine-tuning on Coldplay songs, the model learns:- Poetic language patterns- Common themes (stars, lights, love)- Song structure (verses, chorus)- Coldplay's unique vocabulary

In [ ]:
# Fine-tuning settingssettings_ft = {    "learning_rate": 1e-5,      # Lower LR to preserve pretrained weights    "weight_decay": 0.01,        # Reduced weight decay    "num_epochs": 5,             # Fewer epochs for small dataset    "batch_size": 4,             # Smaller batch size    "warmup_steps": 100,         # Shorter warmup    "max_lr": 1e-5,    "min_lr": 1e-6,    "eval_freq": 50,    "eval_iter": 5,    "gradient_clip": 0.5,        # Gentler clipping    "patience": 3,    "min_improvement": 1e-4,    "print_interval": 1,    "generate_interval": 2}print("Fine-Tuning Settings:")print("=" * 50)print("\nKey differences from pretraining:")print(f"  Learning rate: {settings_ft['learning_rate']} (was {settings['learning_rate']})")print(f"  Epochs: {settings_ft['num_epochs']} (was {settings['num_epochs']})")print(f"  Batch size: {settings_ft['batch_size']} (was {settings['batch_size']})")print(f"  Weight decay: {settings_ft['weight_decay']} (was {settings['weight_decay']})")print("\n✓ Fine-tuning settings configured!")print("\nTo fine-tune:")print("  1. Load your specific dataset (e.g., Coldplay lyrics)")print("  2. Create dataloaders with fine-tuning settings")print("  3. Load pretrained model checkpoint")print("  4. Run train_model() with fine-tuning settings")

## 17. Example Outputs### Before Training (Random):```Input: "I want something"Output: "I want something introduceウ coaches Kard Judaism trendsCommerce rotating infiltration approach"```The model outputs random tokens because weights are randomly initialized.### After Pretraining (IMDb):```Input: "I want something"Output: "I want something that will make me feel better about myself and my life"```The model generates grammatically correct English with reasonable semantics.### After Fine-Tuning (Coldplay):```Input: "Look at the star look how the"Output: "lights go out and the stars begin to fall i hear your voice across the nightlights are running in circles chasing the echoesyou are the star that keeps me aliveOh-ooh-oh-ooh oh, ohi will follow you, i will follow you"```The model captures Coldplay's style: poetic language, themes of stars/lights, emotional tone.### Key Observations:1. **Pretraining teaches language**: Grammar, vocabulary, basic reasoning2. **Fine-tuning teaches style**: Domain-specific patterns, vocabulary, tone3. **Both are necessary**: General knowledge + specific adaptation = useful model

## 18. Summary and Next Steps### What We Built:✅ **Complete GPT Architecture**- Token and positional embeddings- Self-attention mechanism with causal masking- Multi-head attention (8 heads)- Feed-forward networks- Residual connections and layer normalization- 8 transformer decoder blocks✅ **Training Pipeline**- Custom dataset and dataloader- Cross-entropy loss calculation- Cosine learning rate schedule with warmup- Gradient clipping and early stopping- Model checkpointing✅ **Text Generation**- Temperature-based sampling- Top-k filtering- Autoregressive generation### Model Specifications:- **Parameters**: ~10-20M (depending on exact config)- **Context Length**: 256 tokens- **Vocabulary**: 50,257 tokens (GPT-2)- **Attention Heads**: 8- **Layers**: 8 decoder blocks### Next Steps to Improve:1. **Scale Up**:   - Increase model size (more layers, larger dimensions)   - Train on more data (billions of tokens)   - Use larger context length (2048+ tokens)2. **Better Training**:   - Mixed precision training (FP16)   - Distributed training across multiple GPUs   - Better data preprocessing and filtering3. **Advanced Features**:   - Flash Attention for efficiency   - Rotary Position Embeddings (RoPE)   - Group Query Attention (GQA)   - Better tokenization (SentencePiece, BPE)4. **Evaluation**:   - Perplexity metrics   - Downstream task evaluation   - Human evaluation of outputs5. **Deployment**:   - Model quantization (INT8, INT4)   - ONNX export for production   - API serving with FastAPI### Resources for Learning More:- **Papers**:  - "Attention Is All You Need" (Vaswani et al., 2017)  - "Language Models are Unsupervised Multitask Learners" (GPT-2)  - "Language Models are Few-Shot Learners" (GPT-3)- **Code**:  - Hugging Face Transformers library  - nanoGPT by Andrej Karpathy  - PyTorch tutorials- **Courses**:  - Stanford CS224N (NLP with Deep Learning)  - Fast.ai Practical Deep Learning  - DeepLearning.AI courses### Congratulations! 🎉You've built a complete language model from scratch and understand:- How transformers work at a deep level- The training process for LLMs- How to adapt models to specific domainsThis knowledge forms the foundation for working with modern AI systems!